# 01 — Clean the results data

Load the raw international results, drop unplayed fixtures, filter to the modern era, and save the canonical training set.

In [19]:
import pandas as pd
import numpy as np

In [20]:
df = pd.read_csv('../data/raw/international_results/results.csv')
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [21]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 49477 entries, 0 to 49476
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        49477 non-null  str    
 1   home_team   49477 non-null  str    
 2   away_team   49477 non-null  str    
 3   home_score  49449 non-null  float64
 4   away_score  49449 non-null  float64
 5   tournament  49477 non-null  str    
 6   city        49477 non-null  str    
 7   country     49477 non-null  str    
 8   neutral     49477 non-null  bool   
dtypes: bool(1), float64(2), str(6)
memory usage: 5.9 MB


In [22]:
df.isna().sum()

date           0
home_team      0
away_team      0
home_score    28
away_score    28
tournament     0
city           0
country        0
neutral        0
dtype: int64

In [23]:
df[df['home_score'].isna()]

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49449,2026-06-23,Portugal,Uzbekistan,NaN,NaN,FIFA World Cup,Houston,United States,True
49450,2026-06-23,Colombia,DR Congo,NaN,NaN,FIFA World Cup,Zapopan,Mexico,True
49451,2026-06-23,England,Ghana,NaN,NaN,FIFA World Cup,Foxborough,United States,True
49452,2026-06-23,Panama,Croatia,NaN,NaN,FIFA World Cup,Toronto,Canada,True
49453,2026-06-24,Mexico,Czech Republic,NaN,NaN,FIFA World Cup,Mexico City,Mexico,False
49454,2026-06-24,South Africa,South Korea,NaN,NaN,FIFA World Cup,Guadalupe,Mexico,True
49455,2026-06-24,Canada,Switzerland,NaN,NaN,FIFA World Cup,Vancouver,Canada,False
49456,2026-06-24,Bosnia and Herzegovina,Qatar,NaN,NaN,FIFA World Cup,Seattle,United States,True
49457,2026-06-24,Scotland,Brazil,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True
49458,2026-06-24,Morocco,Haiti,NaN,NaN,FIFA World Cup,Atlanta,United States,True


In [24]:
df.duplicated().sum()

np.int64(0)

## Clean & filter

Drop unplayed fixtures, cast types, keep matches from 2006 on, and drop rarely-seen teams.

In [25]:
df = df.dropna(subset=['home_score', 'away_score']).copy()

In [26]:
df['home_score'] = df['home_score'].astype(int)
df['away_score'] = df['away_score'].astype(int)

In [27]:
df["neutral"] = (df["neutral"].astype(str).str.upper()
                 .map({"TRUE": True, "FALSE": False}).fillna(False).astype(bool))

In [28]:
df["date"] = pd.to_datetime(df["date"])

In [29]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 49449 entries, 0 to 49448
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        49449 non-null  datetime64[us]
 1   home_team   49449 non-null  str           
 2   away_team   49449 non-null  str           
 3   home_score  49449 non-null  int64         
 4   away_score  49449 non-null  int64         
 5   tournament  49449 non-null  str           
 6   city        49449 non-null  str           
 7   country     49449 non-null  str           
 8   neutral     49449 non-null  bool          
dtypes: bool(1), datetime64[us](1), int64(2), str(5)
memory usage: 5.4 MB


In [30]:
df = df[df['date'] >= '2006-01-01'].copy()
df.shape

(19717, 9)

In [31]:
min_matches = 30

match_counts = pd.concat([df["home_team"], df["away_team"]]).value_counts()
keep_teams = set(match_counts[match_counts >= min_matches].index)
df = df[df["home_team"].isin(keep_teams) & df["away_team"].isin(keep_teams)].reset_index(drop=True)

## Dixon-Coles assumptions

Quick checks on home advantage, goal correlation, and the neutral-venue split before modeling.

In [32]:
nn = df[~df["neutral"]]
print("Avg home goals:", round(nn["home_score"].mean(), 3))
print("Avg away goals:", round(nn["away_score"].mean(), 3))

Avg home goals: 1.647
Avg away goals: 1.014


In [33]:
print("Corr(home, away goals):", round(df["home_score"].corr(df["away_score"]), 3))
pd.crosstab(df["home_score"].clip(upper=4),
            df["away_score"].clip(upper=4),
            normalize=True).round(3)

Corr(home, away goals): -0.182


away_score,0,1,2,3,4
home_score,,,,,
0,0.090,0.079,0.050,0.025,0.023
1,0.115,0.104,0.052,0.020,0.014
2,0.083,0.077,0.037,0.015,0.006
3,0.049,0.035,0.017,0.004,0.002
4,0.059,0.030,0.010,0.003,0.001


In [34]:
print("Neutral share:", round(df["neutral"].mean(), 3))
neu = df[df["neutral"]]
print("Neutral home-win %:", round((neu["home_score"] > neu["away_score"]).mean(), 3))
print("Neutral away-win %:", round((neu["home_score"] < neu["away_score"]).mean(), 3))

Neutral share: 0.286
Neutral home-win %: 0.405
Neutral away-win %: 0.344


In [35]:
df["tournament"].value_counts().head(10)

tournament
Friendly                                6343
FIFA World Cup qualification            4218
UEFA Euro qualification                 1322
African Cup of Nations qualification    1182
UEFA Nations League                      658
African Cup of Nations                   429
CONCACAF Nations League                  422
AFC Asian Cup qualification              368
FIFA World Cup                           364
Gold Cup                                 275
Name: count, dtype: int64

## Save

In [36]:
df.to_parquet("../data/processed/results_clean.parquet", index=False)